In [4]:
import os
from sedona.spark import SedonaContext
from sedona.spark import dataframe_to_arrow
from sedona.spark.geoarrow import create_spatial_dataframe
from sedona.spark.maps.SedonaKepler import SedonaKepler

In [5]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/22 21:44:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/08/22 21:44:12 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/08/22 21:44:12 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/08/22 21:44:12 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/08/22 21:44:12 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/08/22 21:44:12 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/08/22 21:44:12 WARN SimpleFunctionRegistry: The function st_envelop

# Load input data

In [6]:
import pyspark.sql.functions as f

paris_places = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/paris_places")

# Create H3 cells for Paris

In [7]:
paris_polygon_wkt = "POLYGON((2.2241 48.8156, 2.4699 48.8156, 2.4699 48.9022, 2.2241 48.9022, 2.2241 48.8156))"

In [8]:
h3_cells = sedona.sql(
    f"""
    WITH h3_cells AS (
        SELECT
            id,
            ST_H3ToGeom(ARRAY(id))[0] AS geom
        LATERAL VIEW EXPLODE(ST_H3CellIDs(ST_GeomFromText('{paris_polygon_wkt}'), 8, true)) AS id
    )
    SELECT 
        id,
        geom,
        ST_X(ST_Centroid(geom)) AS lon,
        ST_Y(ST_Centroid(geom)) AS lat
    FROM h3_cells
    """
)

In [9]:
SedonaKepler.create_map(h3_cells, "h3_cells")

/usr/local/lib/python3.10/dist-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


KeplerGl(data={'h3_cells': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20…

In [10]:
h3_cells.createOrReplaceTempView("h3_cells")

# Filter data to categories

In [11]:
categories = [
    'restaurant',
    'shopping',
    'bakery',
    'education',
    'school',
    'pharmacy', 
    'cafe',
    'theatre',
    'transportation',
    'park'
]

paris_places.\
    where(f"categories.primary IN {tuple(categories)}").\
    selectExpr("categories.primary AS category", "geometry").\
    createOrReplaceTempView("selected_categories")

# Create walk catchments

In [9]:
import requests
from shapely.geometry import shape, MultiPolygon

OPEN_ROUTING_URL = "http://host.docker.internal:8085"

def walk_time_polygon(lon: float, lat: float, minutes: int):
    body = {
      "locations": [[lon, lat]],
      "range": [minutes * 60],
      "range_type": "time"
    }

    response = requests.post(
        url=f"{OPEN_ROUTING_URL}/ors/v2/isochrones/foot-walking",
        json=body
    )
    
    if response.status_code != 200:
        return None
        
    response_data = response.json()
    features = response_data["features"]
    shapely_polygons = [
        shape(feature["geometry"]) for feature in features 
        if feature["geometry"]["type"] == 'Polygon'
    ]
    
    return MultiPolygon(shapely_polygons)

In [10]:
import pyspark.sql.functions as f
import sedona.sql.types as st
import shapely.geometry.base as b
 
def create_walk_catchment(
    lon: float,
    lat: float,
    minutes: int
) -> b.BaseGeometry:
    return walk_time_polygon(lon, lat, minutes)
 
create_walk_catchment_udf = f.udf(
    create_walk_catchment,
    st.GeometryType()
)
 
sedona.udf.register(
    "ST_GetWalkCatchment",
    create_walk_catchment_udf
)

In [11]:
sedona.sql(
    """
    WITH catchments AS (
        SELECT
            id,
            ST_GetWalkCatchment(lon, lat, 10) AS walk_catchment,
            geom
        FROM h3_cells
    ),
    joined AS (
        SELECT 
            *,
            c.geom AS grid_geom
        FROM catchments AS c
        JOIN selected_categories AS S ON ST_Intersects(s.geometry, c.walk_catchment)
    )
    SELECT
        id,
        category,
        count(*) AS count,
        FIRST(grid_geom) AS geom
    FROM joined
    GROUP BY id, category
    """
).createOrReplaceTempView("catchments_count")

In [ ]:
sedona.sql("SELECT * FROM catchments_count").show(10)

# Pivot

In [ ]:
feature_df = sedona.table("catchments_count").\
    groupBy("id", "geom").pivot("category", categories).\
    agg(f.first("count")).\
    selectExpr(
        "id",
        "geom",
        "COALESCE(restaurant, 0) AS restaurant",
        "COALESCE(shopping, 0) AS shopping",
        "COALESCE(bakery, 0) AS bakery",
        "COALESCE(education, 0) AS education",
        "COALESCE(school, 0) AS school",
        "COALESCE(pharmacy, 0) AS pharmacy",
        "COALESCE(cafe, 0) AS cafe",
        "COALESCE(theatre, 0) AS theatre",
        "COALESCE(transportation, 0) AS transportation",
        "COALESCE(park, 0) AS park"
    )

In [321]:
feature_df.\
    withColumn("id", f.concat(f.expr("substring(id, 0, 5)"), f.lit("..."))).\
    drop("school", "restaurant", "shopping", "transportation").\
    show(5)

+--------+--------------------+------+---------+--------+----+-------+----+
|      id|                geom|bakery|education|pharmacy|cafe|theatre|park|
+--------+--------------------+------+---------+--------+----+-------+----+
|61304...|POLYGON ((2.35032...|    52|       50|      25|  42|     53|  12|
|61304...|POLYGON ((2.26893...|     5|        3|       7|   3|      1|   5|
|61304...|POLYGON ((2.36062...|    16|       17|      14|  15|     23|   6|
|61304...|POLYGON ((2.47322...|     0|        1|       0|   0|      0|   0|
|61304...|POLYGON ((2.43537...|     0|        0|       0|   0|      8|   4|
+--------+--------------------+------+---------+--------+----+-------+----+
only showing top 5 rows



# Kmeans

## Create features vector

In [337]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

assembler = VectorAssembler(
    inputCols=categories,
    outputCol="features",
)

assembled_df = assembler.transform(feature_df)

In [339]:
assembled_df.select("id", "features").show(5, False)

+------------------+-----------------------------------------------------+
|id                |features                                             |
+------------------+-----------------------------------------------------+
|613047304085045247|[303.0,149.0,52.0,50.0,38.0,25.0,42.0,53.0,16.0,12.0]|
|613047302585581567|[7.0,5.0,5.0,3.0,2.0,7.0,3.0,1.0,2.0,5.0]            |
|613047303986479103|[67.0,24.0,16.0,17.0,15.0,14.0,15.0,23.0,11.0,6.0]   |
|613047304168931327|(10,[3,4],[1.0,1.0])                                 |
|613047304152154111|(10,[0,4,7,9],[2.0,1.0,8.0,4.0])                     |
+------------------+-----------------------------------------------------+
only showing top 5 rows



In [340]:
assembled_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- geom: geometry (nullable = true)
 |-- restaurant: long (nullable = false)
 |-- shopping: long (nullable = false)
 |-- bakery: long (nullable = false)
 |-- education: long (nullable = false)
 |-- school: long (nullable = false)
 |-- pharmacy: long (nullable = false)
 |-- cafe: long (nullable = false)
 |-- theatre: long (nullable = false)
 |-- transportation: long (nullable = false)
 |-- park: long (nullable = false)
 |-- features: vector (nullable = true)



## Train and predict

In [346]:
# Train KMeans model
kmeans = KMeans(k=10, seed=42, featuresCol="features")
model = kmeans.fit(assembled_df)

# Make predictions
predictions = model.transform(assembled_df)
predictions.select("id", "features", "prediction").show(5)

# Show cluster centers
print("Cluster Centers:")
for center in model.clusterCenters():
    print(center)

[Stage 826:>                                                        (0 + 1) / 1]

+------------------+--------------------+----------+
|                id|            features|prediction|
+------------------+--------------------+----------+
|613047304085045247|[303.0,149.0,52.0...|         5|
|613047302585581567|[7.0,5.0,5.0,3.0,...|         0|
|613047303986479103|[67.0,24.0,16.0,1...|         8|
|613047304168931327|(10,[3,4],[1.0,1.0])|         0|
|613047304152154111|(10,[0,4,7,9],[2....|         0|
+------------------+--------------------+----------+
only showing top 5 rows

Cluster Centers:
[4.90769231 1.78461538 1.6        1.21538462 1.28461538 1.63846154
 0.48461538 0.77692308 1.13846154 1.56923077]
[179.    98.5   48.25  52.25  29.75  28.5   35.25  33.75  26.5    9.75]
[146.33333333 101.33333333  16.          63.33333333  42.66666667
  28.33333333  23.          14.          39.           6.33333333]
[47.38709677 16.48387097 13.19354839 12.          9.25806452 11.77419355
  5.74193548  8.12903226  8.35483871  6.83870968]
[143.75    70.3125  38.8125  23.4375  19

In [345]:
SedonaKepler.create_map(predictions, "predictions")

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'predictions': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,…